<a href="https://colab.research.google.com/github/aniketpathak028/fall-detection-system/blob/main/fall_detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="SDTYV08cZM8TUMbHEwcx")
project = rf.workspace("santhoshkumar-v").project("human-fall-e2evv")
version = project.version(2)

dataset = version.download(model_format="yolov8", location="/content/local_dataset")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/local_dataset in yolov8:: 100%|██████████| 10206/10206 [00:01<00:00, 8343.46it/s]


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 8.0 MB/s eta 0:00:00


In [ ]:
# import libraries
from ultralytics import YOLO
import shutil, os

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [ ]:
drive_model_dir = "/content/drive/MyDrive/saved_models"
best_weights_path = os.path.join(drive_model_dir, "fall_detection_best.pt")

if os.path.exists(best_weights_path):
    print(
        f"Weights already found at {best_weights_path}. Skipping training and loading existing model."
    )
    model = YOLO(best_weights_path)
else:
    print("No saved weights found. Starting training...")

    model = YOLO("yolov8n.pt")

    results = model.train(
        data=dataset.location + "/data.yaml",
        epochs=5,
        imgsz=640,
        batch=16,
        workers=2,
        device=0,
    )

    source_dir = os.path.join(results.save_dir, "weights")

    os.makedirs(drive_model_dir, exist_ok=True)

    if os.path.exists(os.path.join(source_dir, "best.pt")):
        shutil.copy(
            os.path.join(source_dir, "best.pt"),
            os.path.join(drive_model_dir, "fall_detection_best.pt"),
        )
        shutil.copy(
            os.path.join(source_dir, "last.pt"),
            os.path.join(drive_model_dir, "fall_detection_last.pt"),
        )
        print("Weights successfully saved to Google Drive!")
    else:
        print("Training completed, but expected weight files were not found.")

    model = YOLO(best_weights_path)

No saved weights found. Starting training...
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/local_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_sc

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/saved_models/fall_detection_best.pt")

# validation
metrics = model.val(data=dataset.location + "/data.yaml")
print(f"mAP50: {metrics.box.map50}")

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1269.3±609.3 MB/s, size: 45.9 KB)
val: Scanning /content/local_dataset/valid/labels.cache... 1021 images, 250 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1021/1021 329.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 64/64 12.1it/s 5.3s
                   all       1021        812      0.779      0.729      0.801      0.493
                Fallen        383        393      0.862      0.732      0.835      0.518
               Falling        391        419      0.697      0.726      0.767      0.469
Speed: 0.7ms preprocess, 1.1ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.8007952839018448
